# 06 — Supervised Ranking Baselines
## Amazon Electronics 5-core · Popularity · MF-BPR · GRU4Rec

## Cell 1 — Imports, Config & Device


In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1 — Imports, config, device
# Run after every kernel restart — everything downstream depends on this.
# ══════════════════════════════════════════════════════════════════════════════

import sys, time, random, math, json
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

# ── Project root ───────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.home() / 'Desktop' / 'Capstone_C2C_Behavioral_Dynamics'
assert PROJECT_ROOT.exists(), f'Project not found: {PROJECT_ROOT}'
assert (PROJECT_ROOT / 'src').exists(), f'src/ missing in {PROJECT_ROOT}'
sys.path.insert(0, str(PROJECT_ROOT))
from src.utils.paths import P

# ── Hyperparameters ────────────────────────────────────────────────────────────
CFG = {
    # Reproducibility
    'SEED'        : 42,
    # Evaluation
    'N_NEG_EVAL'  : 99,
    'TOP_K'       : 10,
    'EVAL_BATCH'  : 512,
    # Training
    'N_EPOCHS'    : 1,          # ← set to 20 for final submission
    'BATCH_SIZE'  : 1024,
    'LR'          : 1e-3,
    'N_NEG_TRAIN' : 4,
    # Architecture
    'EMB_DIM'     : 64,
    'HIDDEN_DIM'  : 128,
    'MAX_SEQ_LEN' : 50,
    # Electronics-specific
    'MIN_RATING'  : 4.0,        # only 5★ reviews are positives — same as UMAG
    'MIN_REVIEWS' : 3,          # drop users with fewer 5★ interactions
    'DIEN_AUC'    : 0.7792,     # benchmark to beat
    'DATASET'     : 'amazon_electronics_5core',
}

# ── Device — single source of truth ───────────────────────────────────────────
DEVICE = torch.device(
    'mps'  if torch.backends.mps.is_available()  else
    'cuda' if torch.cuda.is_available()          else
    'cpu'
)

# ── Seeds ──────────────────────────────────────────────────────────────────────
random.seed(CFG['SEED'])
np.random.seed(CFG['SEED'])
torch.manual_seed(CFG['SEED'])
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(CFG['SEED'])

ELEC_JSON = P.AMAZON_ELEC_DIR / 'reviews_Electronics_5.json'
assert ELEC_JSON.exists(), f'Electronics JSON not found: {ELEC_JSON}'

print(f'Project  : {PROJECT_ROOT}')
print(f'Dataset  : {ELEC_JSON}')
print(f'Device   : {DEVICE}')
print(f'Epochs   : {CFG["N_EPOCHS"]}  (set N_EPOCHS=20 for final run)')
print(f'Positive : rating >= {CFG["MIN_RATING"]} stars only')
print(f'DIEN AUC : {CFG["DIEN_AUC"]}  (benchmark to beat)')
print(f'Metrics  : HR@{CFG["TOP_K"]} · NDCG@{CFG["TOP_K"]} · AUC')
print('Cell 1 ✓')

Project  : /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics
Dataset  : /Volumes/T5 EVO/hf/amazon_electronics/reviews_Electronics_5.json
Device   : cpu
Epochs   : 1  (set N_EPOCHS=20 for final run)
Positive : rating >= 4.0 stars only
DIEN AUC : 0.7792  (benchmark to beat)
Metrics  : HR@10 · NDCG@10 · AUC
Cell 1 ✓


## Cell 2 — Data Load


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2 — Load Electronics JSON  (streaming, orjson-accelerated)
#
# Fixes vs original:
#   1. orjson streaming replaces pd.read_json — faster + lower peak RAM
#   2. Inline rating filter — rejected rows never allocated into df
#   3. (user, item) dedup — keep earliest timestamp
#   4. Sort by (user, timestamp) before indexing
# ══════════════════════════════════════════════════════════════════════════════

import orjson

print('Loading Electronics JSON (streaming, orjson-accelerated)...')
t0 = time.time()

MIN_RATING = CFG['MIN_RATING']
rows  = []
total = 0
skipped = 0

with open(ELEC_JSON, 'rb', buffering=8 * 1024 * 1024) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            rec = orjson.loads(line)
        except Exception:
            skipped += 1
            continue

        uid     = rec.get('reviewerID')
        asin    = rec.get('asin')
        ts      = rec.get('unixReviewTime')
        overall = rec.get('overall')

        if None in (uid, asin, ts, overall):
            skipped += 1
            continue

        total += 1

        # ── Inline rating filter ─────────────────────────────────────────────
        if float(overall) < MIN_RATING:
            continue

        rows.append((uid, asin, float(overall), int(ts)))

print(f'  Parsed {total:,} rows, skipped {skipped:,} malformed  ({time.time()-t0:.1f}s)')

df_raw_len = total
df = pd.DataFrame(rows, columns=['user_id', 'item_id', 'rating', 'timestamp'])
print(f'  Total reviews   : {df_raw_len:,}')
print(f'  After {MIN_RATING:.0f}★ filter : {len(df):,}  ({len(df)/df_raw_len*100:.1f}%)')

# ── Dedup: same (user, item) → keep earliest timestamp ──────────────────────
before = len(df)
df = (df.sort_values('timestamp')
        .drop_duplicates(subset=['user_id', 'item_id'], keep='first')
        .reset_index(drop=True))
print(f'  After dedup     : {len(df):,}  (removed {before - len(df):,} duplicates)')

# ── Sort chronologically ─────────────────────────────────────────────────────
df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)

# ── Integer indices (0 reserved for padding) ─────────────────────────────────
user2idx = {u: i+1 for i, u in enumerate(df['user_id'].unique())}
item2idx = {it: i+1 for i, it in enumerate(df['item_id'].unique())}
df['u'] = df['user_id'].map(user2idx)
df['i'] = df['item_id'].map(item2idx)
N_USERS = len(user2idx) + 1
N_ITEMS = len(item2idx) + 1

# ── Per-user history; drop short sequences ───────────────────────────────────
user_history = df.groupby('u')['i'].apply(list).to_dict()
user_history = {u: v for u, v in user_history.items() if len(v) >= CFG['MIN_REVIEWS']}
valid_users  = list(user_history.keys())

assert df_raw_len > 1_000_000, f'Expected >1M reviews, got {df_raw_len:,}'
print(f'  Valid users  : {len(valid_users):,}  (>={CFG["MIN_REVIEWS"]} {MIN_RATING:.0f}★ interactions)')
print(f'  Unique items : {N_ITEMS-1:,}')
print('Cell 2 ✓')

Loading Electronics JSON (streaming, orjson-accelerated)...
  Parsed 1,689,188 rows, skipped 0 malformed  (19.4s)
  Total reviews   : 1,689,188
  After 4★ filter : 1,356,067  (80.3%)
  After dedup     : 1,356,067  (removed 0 duplicates)
  Valid users  : 181,273  (>=3 4★ interactions)
  Unique items : 62,892
Cell 2 ✓


## Cell 3 — Leave-One-Out Split

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3 — Leave-one-out split
# last interaction  → test positive
# second-to-last    → val positive
# rest              → training sequence
# Identical to UMAG Electronics evaluation protocol.
# ══════════════════════════════════════════════════════════════════════════════

train_seqs    = {u: items[:-2] for u, items in user_history.items()}
val_pos       = {u: items[-2]  for u, items in user_history.items()}
test_pos      = {u: items[-1]  for u, items in user_history.items()}
user_item_set = {u: set(items) for u, items in user_history.items()}

# Popularity counts built here — used in Cell 6, no extra pass needed
pop_counts = defaultdict(int)
for items in train_seqs.values():
    for it in items:
        pop_counts[it] += 1

def sample_negatives(user: int, n: int, exclude: set) -> list:
    negs = []
    while len(negs) < n:
        cands = np.random.randint(1, N_ITEMS, size=n * 3)
        for c in cands:
            if c not in exclude:
                negs.append(int(c))
            if len(negs) == n:
                break
    return negs

print(f'Train : {len(train_seqs):,} users')
print(f'Val   : {len(val_pos):,} users')
print(f'Test  : {len(test_pos):,} users')
print('Cell 3 ✓')

Train : 181,273 users
Val   : 181,273 users
Test  : 181,273 users
Cell 3 ✓


## Cell 4 — Model Definitions

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4 — All model class definitions
# Kept together: one restart → run cells 1-4 → load any checkpoint.
# ══════════════════════════════════════════════════════════════════════════════

# ── MF-BPR ────────────────────────────────────────────────────────────────────
class MFBPR(nn.Module):
    """Matrix Factorisation with Bayesian Personalised Ranking loss."""
    def __init__(self, n_users: int, n_items: int, emb_dim: int):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim, padding_idx=0)
        self.item_emb = nn.Embedding(n_items, emb_dim, padding_idx=0)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)

    def forward(self, users, pos_items, neg_items):
        u  = self.user_emb(users)
        pi = self.item_emb(pos_items)
        ni = self.item_emb(neg_items)
        return -torch.log(torch.sigmoid((u*pi).sum(-1) - (u*ni).sum(-1)) + 1e-8).mean()

    def score(self, users, items):
        return (self.user_emb(users) * self.item_emb(items)).sum(-1)


# ── GRU4Rec ───────────────────────────────────────────────────────────────────
class GRU4Rec(nn.Module):
    """Session-based recommender using a GRU over item embeddings."""
    def __init__(self, n_items: int, emb_dim: int, hidden_dim: int,
                 n_layers: int = 1, dropout: float = 0.2):
        super().__init__()
        self.item_emb = nn.Embedding(n_items, emb_dim, padding_idx=0)
        self.gru      = nn.GRU(emb_dim, hidden_dim, n_layers,
                               batch_first=True,
                               dropout=dropout if n_layers > 1 else 0.0)
        self.out_proj = nn.Linear(hidden_dim, emb_dim)
        self.drop     = nn.Dropout(dropout)
        nn.init.normal_(self.item_emb.weight, std=0.01)

    def encode_seq(self, seq: torch.Tensor) -> torch.Tensor:
        x        = self.drop(self.item_emb(seq))
        out, _   = self.gru(x)
        lengths  = (seq != 0).sum(dim=1).clamp(min=1) - 1
        idx      = lengths.view(-1, 1, 1).expand(-1, 1, out.size(2))
        last     = out.gather(1, idx).squeeze(1)
        return self.out_proj(last)

    def forward(self, seq, pos_items, neg_items):
        u  = self.encode_seq(seq)
        ps = (u * self.item_emb(pos_items)).sum(-1)
        ns = (u * self.item_emb(neg_items)).sum(-1)
        return -torch.log(torch.sigmoid(ps - ns) + 1e-8).mean()

    def score(self, seq, items):
        return (self.encode_seq(seq) * self.item_emb(items)).sum(-1)


print('MFBPR   defined ✓')
print('GRU4Rec defined ✓')
print('Cell 4 ✓')

MFBPR   defined ✓
GRU4Rec defined ✓
Cell 4 ✓


## Cell 5 — Shared Evaluation Engine


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 5 — Shared batched evaluation engine
#
# evaluate_batched() accepts score_fn(users, items) -> np.ndarray
# where users and items are flat lists covering ALL candidates for a full
# batch of users. One call per batch — no Python loop over individual users.
#
# This fixes the original per-user evaluate() which caused ~180 min runtime.
# ══════════════════════════════════════════════════════════════════════════════

def _pad(items: list, max_len: int) -> list:
    seq = items[-max_len:]
    return [0] * (max_len - len(seq)) + seq


def evaluate_batched(
    score_fn,
    split      : str = 'test',
    n_neg      : int = 99,
    k          : int = 10,
    batch_size : int = 512,
) -> dict:
    """
    Batched leave-one-out evaluation.
    HR@k, NDCG@k, AUC computed over (1 pos + n_neg) candidates per user.
    """
    assert split in ('val', 'test')
    pos_dict   = val_pos if split == 'val' else test_pos
    users_list = list(pos_dict.keys())
    hrs, ndcgs, aucs = [], [], []
    t0 = time.time()

    for start in range(0, len(users_list), batch_size):
        elapsed = time.time() - t0
        done    = min(start + batch_size, len(users_list))
        pct     = 100 * done / len(users_list)
        eta     = (elapsed / done * (len(users_list) - done)) if done else 0
        print(f'\r  [{pct:5.1f}%]  {done:,}/{len(users_list):,} users'
              f'  |  elapsed {elapsed:.0f}s  eta {eta:.0f}s   ',
              end='', flush=True)

        batch_users      = users_list[start : start + batch_size]
        all_users        = []
        all_items        = []
        n_cands_per_user = []

        for u in batch_users:
            candidates = [pos_dict[u]] + sample_negatives(u, n_neg, user_item_set[u])
            all_users.extend([u] * len(candidates))
            all_items.extend(candidates)
            n_cands_per_user.append(len(candidates))

        scores = score_fn(all_users, all_items)

        ptr = 0
        for n_cands in n_cands_per_user:
            s         = scores[ptr : ptr + n_cands]
            ptr      += n_cands
            labels    = np.zeros(n_cands, dtype=np.int32)
            labels[0] = 1
            rank      = int((s > s[0]).sum()) + 1
            hrs.append(  1.0 if rank <= k else 0.0)
            ndcgs.append(1.0 / math.log2(rank + 1) if rank <= k else 0.0)
            aucs.append( roc_auc_score(labels, s))

    elapsed = time.time() - t0
    print(f'\r  [100.0%]  {len(users_list):,}/{len(users_list):,} users'
          f'  |  total {elapsed:.1f}s                        ')

    return {
        f'HR@{k}'  : round(float(np.mean(hrs)),   4),
        f'NDCG@{k}': round(float(np.mean(ndcgs)), 4),
        'AUC'      : round(float(np.mean(aucs)),  4),
        'n_users'  : len(hrs),
        'split'    : split,
        'elapsed_s': round(elapsed, 1),
    }


def save_results(results: dict, name: str) -> Path:
    out_dir = P.metrics('electronics')
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f'{name}.json'
    with open(path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'  Saved → {path}')
    return path


def print_results(results: dict, model_name: str):
    k = CFG['TOP_K']
    print(f'  HR@{k}    = {results[f"HR@{k}"]:.4f}')
    print(f'  NDCG@{k}  = {results[f"NDCG@{k}"]:.4f}')
    print(f'  AUC     = {results["AUC"]:.4f}')
    if model_name != 'Popularity':
        delta = results['AUC'] - CFG['DIEN_AUC']
        sign  = '+' if delta >= 0 else ''
        print(f'  vs DIEN : {sign}{delta:.4f}  (target {CFG["DIEN_AUC"]})')
    print(f'  Users   = {results["n_users"]:,}')
    print(f'  Time    = {results["elapsed_s"]}s')


print('evaluate_batched() ready ✓')
print('Cell 5 ✓')

evaluate_batched() ready ✓
Cell 5 ✓


## Cell 6 — Popularity Baseline


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 6 — Popularity baseline
# pop_counts was built in Cell 3 from training sequences.
# ══════════════════════════════════════════════════════════════════════════════

def pop_score_fn(users: list, items: list) -> np.ndarray:
    return np.array([pop_counts.get(it, 0) for it in items], dtype=np.float32)

print('Evaluating Popularity...')
pop_results = evaluate_batched(
    pop_score_fn,
    split      = 'test',
    n_neg      = CFG['N_NEG_EVAL'],
    k          = CFG['TOP_K'],
    batch_size = CFG['EVAL_BATCH'],
)
pop_results['model']   = 'Popularity'
pop_results['epochs']  = 0
pop_results['dataset'] = CFG['DATASET']

print_results(pop_results, 'Popularity')
save_results(pop_results, 'popularity')
print('Cell 6 ✓')

Evaluating Popularity...
  [100.0%]  181,273/181,273 users  |  total 200.0s                        
  HR@10    = 0.5438
  NDCG@10  = 0.3549
  AUC     = 0.7883
  Users   = 181,273
  Time    = 200.0s
  Saved → /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/results/metrics/electronics/popularity.json
Cell 6 ✓


## Cell 7 — MF-BPR Training


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 7 — MF-BPR training
# Skips training if checkpoint already exists.
# ══════════════════════════════════════════════════════════════════════════════

MF_CKPT = P.ckpt('mf_bpr_electronics') / f'mf_bpr_epoch{CFG["N_EPOCHS"]}.pt'

if MF_CKPT.exists():
    print(f'Checkpoint found → skipping training')
    print(f'  {MF_CKPT}')
    print('Delete the file above to force re-training.')
else:
    class BPRDataset(Dataset):
        def __init__(self, train_seqs, user_item_set, n_items, n_neg=4):
            self.users         = list(train_seqs.keys())
            self.train_seqs    = train_seqs
            self.user_item_set = user_item_set
            self.n_items       = n_items
            self.n_neg         = n_neg

        def __len__(self):
            return len(self.users) * self.n_neg

        def __getitem__(self, idx):
            u   = self.users[idx % len(self.users)]
            pos = random.choice(self.train_seqs[u])
            neg = random.randint(1, self.n_items - 1)
            while neg in self.user_item_set[u]:
                neg = random.randint(1, self.n_items - 1)
            return torch.tensor(u), torch.tensor(pos), torch.tensor(neg)

    mf_model  = MFBPR(N_USERS, N_ITEMS, CFG['EMB_DIM']).to(DEVICE)
    mf_optim  = torch.optim.Adam(mf_model.parameters(), lr=CFG['LR'])
    mf_loader = DataLoader(
        BPRDataset(train_seqs, user_item_set, N_ITEMS, CFG['N_NEG_TRAIN']),
        batch_size=CFG['BATCH_SIZE'], shuffle=True, num_workers=0,
    )

    print(f'Training MF-BPR — {CFG["N_EPOCHS"]} epoch(s), {len(mf_loader):,} batches...')
    for epoch in range(CFG['N_EPOCHS']):
        mf_model.train()
        total_loss, t0 = 0.0, time.time()
        for step, (u, pos, neg) in enumerate(mf_loader):
            mf_optim.zero_grad()
            loss = mf_model(u.to(DEVICE), pos.to(DEVICE), neg.to(DEVICE))
            loss.backward()
            mf_optim.step()
            total_loss += loss.item()
            if (step + 1) % 200 == 0:
                print(f'  step {step+1:,}/{len(mf_loader):,}'
                      f' | loss {total_loss/(step+1):.4f}')
        print(f'  Epoch {epoch+1} | {time.time()-t0:.1f}s'
              f' | avg loss {total_loss/len(mf_loader):.4f}')

    MF_CKPT.parent.mkdir(parents=True, exist_ok=True)
    torch.save(mf_model.state_dict(), MF_CKPT)
    print(f'Checkpoint saved → {MF_CKPT}')

print('Cell 7 ✓')

Training MF-BPR — 1 epoch(s), 709 batches...
  step 200/709 | loss 0.6927
  step 400/709 | loss 0.6916
  step 600/709 | loss 0.6890
  Epoch 1 | 87.5s | avg loss 0.6868
Checkpoint saved → /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/results/checkpoints/mf_bpr_electronics/mf_bpr_epoch1.pt
Cell 7 ✓


## Cell 8 — MF-BPR Evaluation


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 8 — MF-BPR evaluation from checkpoint
# ══════════════════════════════════════════════════════════════════════════════

MF_CKPT = P.ckpt('mf_bpr_electronics') / f'mf_bpr_epoch{CFG["N_EPOCHS"]}.pt'
assert MF_CKPT.exists(), f'Checkpoint not found: {MF_CKPT}\nRun Cell 7 first.'

mf_model = MFBPR(N_USERS, N_ITEMS, CFG['EMB_DIM']).to(DEVICE)
mf_model.load_state_dict(torch.load(MF_CKPT, map_location=DEVICE))
mf_model.eval()
print(f'Loaded: {MF_CKPT}')

def mf_score_fn(users: list, items: list) -> np.ndarray:
    with torch.no_grad():
        u = torch.tensor(users, dtype=torch.long, device=DEVICE)
        i = torch.tensor(items, dtype=torch.long, device=DEVICE)
        return mf_model.score(u, i).cpu().numpy()

print('Evaluating MF-BPR...')
mf_results = evaluate_batched(
    mf_score_fn,
    split      = 'test',
    n_neg      = CFG['N_NEG_EVAL'],
    k          = CFG['TOP_K'],
    batch_size = CFG['EVAL_BATCH'],
)
mf_results['model']      = 'MF-BPR'
mf_results['epochs']     = CFG['N_EPOCHS']
mf_results['dataset']    = CFG['DATASET']
mf_results['checkpoint'] = str(MF_CKPT)

print_results(mf_results, 'MF-BPR')
save_results(mf_results, f'mf_bpr_epoch{CFG["N_EPOCHS"]}')
print('Cell 8 ✓')

Loaded: /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/results/checkpoints/mf_bpr_electronics/mf_bpr_epoch1.pt
Evaluating MF-BPR...
  [100.0%]  181,273/181,273 users  |  total 284.8s                        
  HR@10    = 0.1986
  NDCG@10  = 0.1230
  AUC     = 0.5067
  vs DIEN : -0.2725  (target 0.7792)
  Users   = 181,273
  Time    = 284.8s
  Saved → /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/results/metrics/electronics/mf_bpr_epoch1.json
Cell 8 ✓


## Cell 9 — GRU4Rec Training


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 9 — GRU4Rec training  (progress bar + speed-ups)
# Skips training if checkpoint already exists.
# ══════════════════════════════════════════════════════════════════════════════
import random
import time
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm 

# ── Dataset defined at MODULE level (required for multiprocessing pickle) ─────
# Even with num_workers=0 this must stay outside the if/else block.
class GRU4RecDataset(Dataset):
    """
    BPR-style dataset.
    Assumption: every user has ≥ 1 interaction in train_seqs.
    """
    def __init__(self, train_seqs, user_item_set, n_items, max_len, n_neg=4):
        self.users         = list(train_seqs.keys())
        self.train_seqs    = train_seqs
        self.user_item_set = user_item_set
        self.n_items       = n_items
        self.max_len       = max_len
        self.n_neg         = n_neg

    def __len__(self):
        return len(self.users) * self.n_neg

    def _pad(self, items):
        seq = items[-self.max_len:]
        return torch.tensor(
            [0] * (self.max_len - len(seq)) + seq, dtype=torch.long
        )

    def _sample_neg(self, seen: set) -> int:
        candidates = torch.randint(1, self.n_items, (32,)).tolist()
        for c in candidates:
            if c not in seen:
                return c
        neg = random.randint(1, self.n_items - 1)
        while neg in seen:
            neg = random.randint(1, self.n_items - 1)
        return neg

    def __getitem__(self, idx):
        u     = self.users[idx % len(self.users)]
        items = self.train_seqs[u]
        t     = random.randint(1, len(items) - 1) if len(items) >= 2 else 0
        seq   = self._pad(items[:t])
        pos   = items[t]
        neg   = self._sample_neg(self.user_item_set[u])
        return seq, torch.tensor(pos, dtype=torch.long), torch.tensor(neg, dtype=torch.long)


# ── Training ──────────────────────────────────────────────────────────────────
GRU_CKPT = P.ckpt('gru4rec_electronics') / f'gru4rec_epoch{CFG["N_EPOCHS"]}.pt'

if GRU_CKPT.exists():
    print(f'Checkpoint found → skipping training')
    print(f'  {GRU_CKPT}')
    print('Delete the file above to force re-training.')
else:
    gru_model = GRU4Rec(N_ITEMS, CFG['EMB_DIM'], CFG['HIDDEN_DIM']).to(DEVICE)

    if hasattr(torch, 'compile'):
        gru_model = torch.compile(gru_model)

    gru_optim = torch.optim.Adam(gru_model.parameters(), lr=CFG['LR'])

    # num_workers=0: required on macOS — MPS + multiprocessing DataLoader
    # causes spawn failures. MPS transfers are async so there's no CPU stall.
    # persistent_workers only valid when num_workers > 0, so omitted here.
    gru_loader = DataLoader(
        GRU4RecDataset(train_seqs, user_item_set, N_ITEMS,
                       CFG['MAX_SEQ_LEN'], CFG['N_NEG_TRAIN']),
        batch_size  = CFG['BATCH_SIZE'],
        shuffle     = True,
        num_workers = 0,
    )

    print(f'Training GRU4Rec — {CFG["N_EPOCHS"]} epoch(s), '
          f'{len(gru_loader):,} batches/epoch')

    for epoch in range(CFG['N_EPOCHS']):
        gru_model.train()
        total_loss = 0.0
        t0 = time.time()

        pbar = tqdm(
            gru_loader,
            desc         = f'Epoch {epoch+1}/{CFG["N_EPOCHS"]}',
            unit         = 'batch',
            dynamic_ncols= True,
            leave        = True,
        )

        for step, (seqs, pos, neg) in enumerate(pbar, start=1):
            gru_optim.zero_grad()
            loss = gru_model(seqs.to(DEVICE), pos.to(DEVICE), neg.to(DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(gru_model.parameters(), 1.0)
            gru_optim.step()

            total_loss += loss.item()
            pbar.set_postfix(loss=f'{total_loss/step:.4f}', refresh=False)

        elapsed = time.time() - t0
        print(f'  → Epoch {epoch+1} done | {elapsed:.1f}s '
              f'| avg loss {total_loss/len(gru_loader):.4f} '
              f'| {len(gru_loader)/elapsed:.1f} batches/s')

    GRU_CKPT.parent.mkdir(parents=True, exist_ok=True)
    torch.save(gru_model.state_dict(), GRU_CKPT)
    print(f'Checkpoint saved → {GRU_CKPT}')

print('Cell 9 ✓')

Training GRU4Rec — 1 epoch(s), 709 batches/epoch


Epoch 1/1: 100%|██████████| 709/709 [2:32:47<00:00, 12.93s/batch, loss=0.4422]  

  → Epoch 1 done | 9167.6s | avg loss 0.4422 | 0.1 batches/s
Checkpoint saved → /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/results/checkpoints/gru4rec_electronics/gru4rec_epoch1.pt
Cell 9 ✓


## Cell 10 — GRU4Rec Evaluation


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 10 — GRU4Rec evaluation from checkpoint
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np
import torch
from tqdm import tqdm

GRU_CKPT = P.ckpt('gru4rec_electronics') / f'gru4rec_epoch{CFG["N_EPOCHS"]}.pt'
assert GRU_CKPT.exists(), f'Checkpoint not found: {GRU_CKPT}\nRun Cell 9 first.'

# ── Load model — strip _orig_mod. prefix added by torch.compile ───────────────
gru_model = GRU4Rec(N_ITEMS, CFG['EMB_DIM'], CFG['HIDDEN_DIM']).to(DEVICE)

state_dict = torch.load(GRU_CKPT, map_location=DEVICE)
if any(k.startswith('_orig_mod.') for k in state_dict):
    state_dict = {k.replace('_orig_mod.', '', 1): v for k, v in state_dict.items()}

gru_model.load_state_dict(state_dict)
gru_model.eval()
print(f'Loaded: {GRU_CKPT}')

# ── Standalone pad helper (no dependency on GRU4RecDataset instance) ──────────
# Defined here so Cell 10 is self-contained even after a kernel restart.
def _pad_seq(items: list, max_len: int) -> torch.Tensor:
    seq = items[-max_len:]
    return torch.tensor(
        [0] * (max_len - len(seq)) + seq, dtype=torch.long
    )

# ── Build padded sequence cache as a stacked tensor ──────────────────────────
print('Building sequence cache...')

eval_users  = list(test_pos.keys())
user_to_idx = {u: i for i, u in enumerate(eval_users)}   # O(1) lookup

seq_matrix = torch.stack([
    _pad_seq(train_seqs.get(u, [1]), CFG['MAX_SEQ_LEN'])
    for u in tqdm(eval_users, desc='Padding seqs', leave=False)
], dim=0).to(DEVICE)                  # (n_users, MAX_SEQ_LEN) — lives on device

print(f'  Sequence cache: {seq_matrix.shape} '
      f'| {seq_matrix.element_size() * seq_matrix.nelement() / 1e6:.1f} MB on {DEVICE}')

# ── Score function ────────────────────────────────────────────────────────────
@torch.no_grad()
def gru_score_fn(users: list, items: list) -> np.ndarray:
    row_idx = torch.tensor(
        [user_to_idx[u] for u in users],
        dtype=torch.long, device=DEVICE
    )
    seqs_t  = seq_matrix[row_idx]                                    # (B, MAX_SEQ_LEN)
    items_t = torch.tensor(items, dtype=torch.long, device=DEVICE)  # (B,)
    return gru_model.score(seqs_t, items_t).cpu().numpy()

# ── Evaluate ──────────────────────────────────────────────────────────────────
print('Evaluating GRU4Rec...')
gru_results = evaluate_batched(
    gru_score_fn,
    split      = 'test',
    n_neg      = CFG['N_NEG_EVAL'],
    k          = CFG['TOP_K'],
    batch_size = CFG['EVAL_BATCH'],
)

gru_results['model']      = 'GRU4Rec'
gru_results['epochs']     = CFG['N_EPOCHS']
gru_results['dataset']    = CFG['DATASET']
gru_results['checkpoint'] = str(GRU_CKPT)

print_results(gru_results, 'GRU4Rec')
save_results(gru_results,  f'gru4rec_epoch{CFG["N_EPOCHS"]}')
print('Cell 10 ✓')

Loaded: /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/results/checkpoints/gru4rec_electronics/gru4rec_epoch1.pt
Building sequence cache...


  Sequence cache: torch.Size([181273, 50]) | 72.5 MB on cpu
Evaluating GRU4Rec...
  [100.0%]  181,273/181,273 users  |  total 12935.7s                        
  HR@10    = 0.5238
  NDCG@10  = 0.3428
  AUC     = 0.7783
  vs DIEN : -0.0009  (target 0.7792)
  Users   = 181,273
  Time    = 12935.7s
  Saved → /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/results/metrics/electronics/gru4rec_epoch1.json
Cell 10 ✓


## Cell 12 — Final Results Table


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 12 — Results table vs DIEN benchmark
# Loads all results from disk — fully independent of kernel state.
# ══════════════════════════════════════════════════════════════════════════════

import json, math
from pathlib import Path

METRICS_DIR = P.metrics('electronics')
EPOCH       = CFG['N_EPOCHS']

def load_result(filename: str) -> dict | None:
    path = METRICS_DIR / filename
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return None

results = {
    'Popularity' : load_result('popularity.json'),
    'MF-BPR'     : load_result(f'mf_bpr_epoch{EPOCH}.json'),
    'GRU4Rec'    : load_result(f'gru4rec_epoch{EPOCH}.json'),
    'UMAG v6'    : load_result(f'umag_epoch{EPOCH}.json'),
}

k = CFG['TOP_K']

def _fmt(r, key):
    if r is None: return 'pending'
    v = r.get(key)
    if v is None or (isinstance(v, float) and math.isnan(v)): return 'pending'
    return f'{v:.4f}'

def _delta(r):
    if r is None: return ''
    v = r.get('AUC')
    if v is None: return ''
    d = v - CFG['DIEN_AUC']
    return f'  ({"+" if d >= 0 else ""}{d:.4f} vs DIEN)'

notes = {
    'Popularity' : 'no personalization, no history',
    'MF-BPR'     : 'no sequential history, no gate',
    'GRU4Rec'    : 'no gate, no tier-awareness',
    'UMAG v6'    : 'full model  ← target',
}

print(f'Epochs   : {EPOCH}  (set N_EPOCHS=20 for final submission)')
print(f'Dataset  : Amazon Electronics 5-core (5★ only)')
print(f'Protocol : leave-one-out · {CFG["N_NEG_EVAL"]} negatives · HR@{k} · NDCG@{k} · AUC')
print(f'Benchmark: DIEN AUC = {CFG["DIEN_AUC"]}')
print()
print(f'{"Model":<14} {f"HR@{k}":>8} {f"NDCG@{k}":>10} {"AUC":>8}  Note')
print('─' * 78)
for name, r in results.items():
    print(f'{name:<14} {_fmt(r, f"HR@{k}"):>8} {_fmt(r, f"NDCG@{k}"):>10}'
          f' {_fmt(r, "AUC"):>8}  {notes[name]}{_delta(r)}')
print('─' * 78)

# ── Save combined CSV ──────────────────────────────────────────────────────────
rows = []
for name, r in results.items():
    rows.append({
        'model'     : name,
        f'HR@{k}'   : None if r is None else r.get(f'HR@{k}'),
        f'NDCG@{k}' : None if r is None else r.get(f'NDCG@{k}'),
        'AUC'       : None if r is None else r.get('AUC'),
        'vs_dien'   : None if r is None or r.get('AUC') is None
                      else round(r['AUC'] - CFG['DIEN_AUC'], 4),
        'epochs'    : EPOCH,
        'dataset'   : CFG['DATASET'],
    })

out_csv = METRICS_DIR / f'all_baselines_epoch{EPOCH}.csv'
pd.DataFrame(rows).to_csv(out_csv, index=False)
print(f'\nSaved combined CSV → {out_csv}')
print('Cell 12 ✓')

Epochs   : 1  (set N_EPOCHS=20 for final submission)
Dataset  : Amazon Electronics 5-core (5★ only)
Protocol : leave-one-out · 99 negatives · HR@10 · NDCG@10 · AUC
Benchmark: DIEN AUC = 0.7792

Model             HR@10    NDCG@10      AUC  Note
──────────────────────────────────────────────────────────────────────────────
Popularity       0.5438     0.3549   0.7883  no personalization, no history  (+0.0091 vs DIEN)
MF-BPR           0.1986     0.1230   0.5067  no sequential history, no gate  (-0.2725 vs DIEN)
GRU4Rec          0.5238     0.3428   0.7783  no gate, no tier-awareness  (-0.0009 vs DIEN)
UMAG v6         pending    pending  pending  full model  ← target
──────────────────────────────────────────────────────────────────────────────

Saved combined CSV → /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/results/metrics/electronics/all_baselines_epoch1.csv
Cell 12 ✓
